In [1]:
import sys
import os
from pathlib import Path
# Go up until you find the project root
while not (Path.cwd() / ".gitignore").exists() and Path.cwd() != Path("/"):
    os.chdir("..")


In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd

In [70]:
df_po_train = pd.read_csv("data/raw/geolifeclef-2025/GLC25_P0_metadata_train.csv")
df_pa_train = pd.read_csv("data/raw/geolifeclef-2025/GLC25_PA_metadata_train.csv")
df_pa_test = pd.read_csv('data/raw/geolifeclef-2025/GLC25_SOLUTION_FILE-v2.csv')

In [106]:
df_po_train['speciesId'] = df_po_train['speciesId'].astype(int)
#species_pa_test = df_pa_test['predictions'].astype(int).unique()
species_pa_train = df_pa_train['speciesId'].astype(int).unique()
species_po_train = df_po_train['speciesId'].astype(int).unique()
# species_all = list(set(species_pa_test)&set(species_pa_train)&set(species_po_train))

In [116]:
from torch.utils.data import Dataset
objective_species = [3383, 1152, 6772, 5300, 7090]
df_pa_train_copy = df_pa_train.copy()[['surveyId','speciesId']]
df_pa_train_copy = df_pa_train_copy[df_pa_train_copy['speciesId'].isin(objective_species)]
class ObservationDataset(Dataset):
    def __init__(self, df, objective_species):
        self.df = df
        self.objective_species = objective_species
    def __len__(self):
        return len(self.df)
    def __getitem__(self,idx):
        sub_df = self.df[self.df['surveyId'] == idx]
        return pd.crosstab(sub_df['surveyId'], sub_df['speciesId']).rename_axis(columns=None).reset_index()

In [120]:
df_o = ObservationDataset(df_pa_train_copy, objective_species)
df_o.__getitem__(38161)

,surveyId,6772.0
0,38161,4


In [87]:
df_po_train_filtered = df_po_train[df_po_train['speciesId'].isin(objective_species)]
po_train_matrix = pd.crosstab(df_po_train_filtered['surveyId'], df_po_train_filtered['speciesId']).rename_axis(columns=None).reset_index()

In [103]:
# all survey ids (even those without any objective species)
all_surveys = df_po_train['surveyId'].unique()

# build counts only for objective species
po_train_matrix = (
    df_po_train.loc[df_po_train['speciesId'].isin(objective_species), ['surveyId', 'speciesId']]
      .assign(_n=1)
      .groupby(['surveyId', 'speciesId'])['_n'].sum()   # count per (survey, species)
      .unstack(fill_value=0)                             # wide format
      .reindex(index=all_surveys,                       # keep all surveys
               columns=objective_species,               # keep only target species (and in this order)
               fill_value=0)
      .astype('Sparse[int]')                            # optional: memory-friendly sparse ints
      .reset_index()
)


In [105]:
po_train_matrix

speciesId,surveyId,3383,1152,6772,5300,7090
0,1,1,0,0,0,0
1,2,0,1,0,0,0
2,3,0,0,1,0,0
3,4,0,0,0,0,0
4,5,0,0,0,0,0
...,...,...,...,...,...,...
3845528,3919658,0,0,0,1,0
3845529,3919659,0,0,0,0,0
3845530,3919660,0,0,0,0,0
3845531,3919661,0,0,0,0,0


In [ ]:
df_po_train.pivot_table(index = 'surveyId', columns = 'speciesId',
                         aggfunc = 'size', fill_value = 0
                        ).rename_axis(columns=None).reset_index()

C:\Users\pablo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pandas\core\reshape\reshape.py:143: RuntimeWarning: overflow encountered in scalar multiply
  num_cells = num_rows * num_columns


ValueError: negative dimensions are not allowed

In [ ]:
df_po_train.pivot_table(index = 'surveyId', columns = 'speciesId', aggfunc = 'size', fill_value = 0)

In [ ]:
df_pa_test['predictions'] = df_pa_test['predictions'].str.split(' ')
df_pa_test['new_test'] = df_pa_test['surveyId'] >= 5000000
solutions_explode = df_pa_test.explode('predictions', ignore_index=True)

In [ ]:
# print('Species PA test: ',len(species_pa_test))
# print('Species PA train: ',len(species_pa_train))
# print('Species PO train: ',len(species_po_train))
# print('Species in all:', len(species_all))

Species PA test:  3986
Species PA train:  5016
Species PO train:  9709
Species in all: 3331


In [ ]:
env_path = os.fspath('data/raw/geolifeclef-2025/EnvironmentalValues/')

'data/raw/geolifeclef-2025/EnvironmentalValues/'

In [23]:
### Environmental Variables
df_env = {}
env_features = ['Elevation', 'HumanFootprint', 'LandCover', 'SoilGrids']
observation = ['PO-train', 'PA-train', 'PA-test'] 
env_path = os.fspath('data/raw/geolifeclef-2025/EnvironmentalValues/')

for e_f in env_features:
    feature_path = os.path.join(env_path, e_f)
    file_list = os.listdir(feature_path)
    for f in file_list:
        for o in observation:
            if o in f:
                df_env[(o,e_f)] = pd.read_csv(os.path.join(feature_path,f))